In [35]:
import numpy as np
import pandas as pd

In [3]:
data = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv'

!curl $data -o data-hw-3.csv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  272k  100  272k    0     0  1591k      0 --:--:-- --:--:-- --:--:-- 1601k


In [43]:
df = pd.read_csv('data-hw-3.csv')

In [4]:
df

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1
...,...,...,...,...,...,...,...,...,...
4995,organic_search,education,employed,asia,45834.0,2,2,0.37,1
4996,NaN,technology,employed,north_america,84831.0,2,5,0.51,0
4997,organic_search,finance,employed,asia,53636.0,4,6,0.75,1
4998,referral,technology,employed,africa,69430.0,4,8,0.79,1


In [91]:
df.head().T

,0,1,2,3,4
lead_source,organic_search,social_media,referral,paid_ads,referral
industry,technology,technology,retail,manufacturing,manufacturing
employment_status,employed,employed,employed,student,student
location,europe,south_america,europe,NA,north_america
annual_income,88160.0,72688.0,44697.0,0.0,24062.0
number_of_courses_viewed,3,3,3,3,4
interaction_count,4,7,4,6,5
lead_score,0.64,0.72,0.58,0.57,0.62
converted,1,1,1,0,1


In [92]:
categorical_columns = list(df.dtypes[df.dtypes == 'object'].index)

In [93]:
categorical_columns

['lead_source', 'industry', 'employment_status', 'location']

In [46]:
for c in categorical_columns:
    df[c] = df[c].str.lower().str.replace(' ', '_')

In [47]:
df.dtypes

lead_source                  object
industry                     object
employment_status            object
location                     object
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [94]:
df.isnull().sum()

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

In [49]:
df.annual_income = pd.to_numeric(df.annual_income, errors='coerce')
df.annual_income = df.annual_income.fillna(0.0)

In [51]:
df.lead_score = pd.to_numeric(df.lead_score, errors='coerce')
df.lead_score = df.lead_score.fillna(0.0)

In [ ]:
df.lead_score = pd.to_numeric(df.lead_score, errors='coerce')
df.lead_score = df.lead_score.fillna(0.0)

In [60]:
df.lead_source = df.lead_source.fillna('NA')


In [54]:
df.employment_status = df.employment_status.fillna('NA')

In [55]:
df.location = df.location.fillna('NA')

In [56]:
# Question 1
df.mode()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,north_america,0.0,2,4,0.46,1


In [57]:
numerical = ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']

In [25]:
numerical

['annual_income',
 'number_of_courses_viewed',
 'interaction_count',
 'lead_score']

In [62]:
# Question 2

df[numerical].corrwith(df.converted)

annual_income               0.189652
number_of_courses_viewed    0.395038
interaction_count           0.448624
lead_score                  0.483246
dtype: float64

In [30]:
df[numerical].corrwith(df.annual_income)

annual_income               1.000000
number_of_courses_viewed    0.161300
interaction_count           0.122842
lead_score                  0.229496
dtype: float64

In [31]:
df[numerical].corrwith(df.number_of_courses_viewed)

annual_income               0.161300
number_of_courses_viewed    1.000000
interaction_count           0.721609
lead_score                  0.757204
dtype: float64

In [32]:
df[numerical].corrwith(df.interaction_count)

annual_income               0.122842
number_of_courses_viewed    0.721609
interaction_count           1.000000
lead_score                  0.915746
dtype: float64

In [33]:
df[numerical].corrwith(df.lead_score)

annual_income               0.229496
number_of_courses_viewed    0.757204
interaction_count           0.915746
lead_score                  1.000000
dtype: float64

In [83]:
from sklearn.model_selection import train_test_split

In [107]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

In [96]:
len(df_train), len(df_val), len(df_test)

(3000, 1000, 1000)

In [108]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [98]:
df_train.head().T

,0,1,2,3,4
lead_source,paid_ads,events,referral,organic_search,events
industry,finance,finance,NA,finance,technology
employment_status,employed,employed,employed,employed,employed
location,asia,south_america,asia,north_america,north_america
annual_income,65980.0,71662.0,40220.0,88352.0,89316.0
number_of_courses_viewed,2,0,2,2,1
interaction_count,3,0,5,2,5
lead_score,0.37,0.09,0.55,0.45,0.0
converted,1,0,1,0,1


In [87]:
y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

In [109]:
del df_train['converted']
del df_val['converted']
del df_test['converted']

In [19]:
df_train.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score
0,paid_ads,finance,employed,asia,65980.0,2,3,0.37
1,events,finance,employed,south_america,71662.0,0,0,0.09
2,referral,NaN,employed,asia,40220.0,2,5,0.55
3,organic_search,finance,employed,north_america,88352.0,2,2,0.45
4,events,technology,employed,north_america,89316.0,1,5,NaN


In [46]:
from sklearn.metrics import mutual_info_score

In [51]:
# Question 3

round(mutual_info_score(y_train, df_train.lead_source), 2)

0.03

In [52]:
round(mutual_info_score(y_train, df_train.industry), 2)

0.0

In [53]:
round(mutual_info_score(y_train, df_train.industry), 3)

0.002

In [54]:
round(mutual_info_score(y_train, df_train.employment_status), 2)

0.02

In [55]:
round(mutual_info_score(y_train, df_train.location), 2)

0.0

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
# solver='lbfgs' is the default solver in newer version of sklearn
# for older versions, you need to specify it explicitly
#model.fit(X_train, y_train)

In [123]:
dicts = df[categorical + numerical].to_dict(orient='records')

In [124]:
categorical = categorical_columns

In [125]:
from sklearn.feature_extraction import DictVectorizer

dv = DictVectorizer(sparse=False)

train_dict = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

In [21]:
dv.get_feature_names_out()

array(['annual_income', 'employment_status', 'employment_status=employed',
       'employment_status=self_employed', 'employment_status=student',
       'employment_status=unemployed', 'industry', 'industry=education',
       'industry=finance', 'industry=healthcare',
       'industry=manufacturing', 'industry=retail', 'industry=technology',
       'interaction_count', 'lead_score', 'lead_source',
       'lead_source=events', 'lead_source=organic_search',
       'lead_source=paid_ads', 'lead_source=referral',
       'lead_source=social_media', 'location', 'location=africa',
       'location=asia', 'location=europe', 'location=north_america',
       'location=south_america', 'number_of_courses_viewed'], dtype=object)

In [23]:
X_train.shape

(3000, 28)

In [126]:
val_dict = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dict)

In [25]:
X_val.shape

(1000, 28)

In [127]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
# solver='lbfgs' is the default solver in newer version of sklearn
# for older versions, you need to specify it explicitly
model.fit(X_train, y_train)

LogisticRegression(max_iter=1000, random_state=42, solver='liblinear')

In [74]:
model.intercept_[0]

-0.010838978497809288

In [75]:
model.coef_[0].round(3)

array([-0.   , -0.001,  0.007, -0.003, -0.008, -0.006, -0.   , -0.003,
       -0.001, -0.002, -0.002, -0.002, -0.001,  0.137,  0.011,  0.   ,
       -0.009,  0.007, -0.014,  0.009, -0.004,  0.   , -0.002, -0.002,
       -0.001, -0.003, -0.004,  0.054])

In [41]:
y_train.shape

(3000,)

In [128]:
y_pred = model.predict_proba(X_val)[:, 1]

In [129]:
converted_decision = (y_pred >= 0.5)

In [132]:
model_pred = (y_val == converted_decision).mean()

In [133]:
# Question 4 answer 
full_model_pred

0.645

In [134]:
df_train_copy = df_train.copy()
df_val_copy = df_val.copy()
df_test_copy = df_test.copy()

In [114]:
df_train.head().T

,0,1,2,3,4
lead_source,paid_ads,events,referral,organic_search,events
industry,finance,finance,NA,finance,technology
employment_status,employed,employed,employed,employed,employed
location,asia,south_america,asia,north_america,north_america
annual_income,65980.0,71662.0,40220.0,88352.0,89316.0
number_of_courses_viewed,2,0,2,2,1
interaction_count,3,0,5,2,5
lead_score,0.37,0.09,0.55,0.45,0.0


In [135]:
del df_train_copy['lead_source']
del df_val_copy['lead_source']
del df_test_copy['lead_source']

In [136]:
categorical_copy = ['industry', 'employment_status', 'location']
numerical_copy = numerical

In [137]:
dv = DictVectorizer(sparse=False)

train_dict = df_train_copy[categorical_copy + numerical_copy].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

In [118]:
dv.get_feature_names_out()

array(['annual_income', 'employment_status=NA',
       'employment_status=employed', 'employment_status=self_employed',
       'employment_status=student', 'employment_status=unemployed',
       'industry=NA', 'industry=education', 'industry=finance',
       'industry=healthcare', 'industry=manufacturing', 'industry=retail',
       'industry=technology', 'interaction_count', 'lead_score',
       'location=NA', 'location=africa', 'location=asia',
       'location=europe', 'location=north_america',
       'location=south_america', 'number_of_courses_viewed'], dtype=object)

In [138]:
val_dict = df_val_copy[categorical_copy + numerical_copy].to_dict(orient='records')
X_val = dv.transform(val_dict)

In [139]:
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
# solver='lbfgs' is the default solver in newer version of sklearn
# for older versions, you need to specify it explicitly
model.fit(X_train, y_train)

LogisticRegression(max_iter=1000, random_state=42, solver='liblinear')

In [140]:
y_pred = model.predict_proba(X_val)[:, 1]
converted_decision = (y_pred >= 0.5)
no_lead_model_pred = round((y_val == converted_decision).mean(), 3)

In [142]:
no_lead_result = full_model_pred - no_lead_model_pred

In [143]:
no_lead_result

0.0030000000000000027

In [145]:
# Question 5

df_train_copy = df_train.copy()
df_val_copy = df_val.copy()
df_test_copy = df_test.copy()

del df_train_copy['number_of_courses_viewed']
del df_val_copy['number_of_courses_viewed']
del df_test_copy['number_of_courses_viewed']

categorical_copy = categorical
numerical_copy = ['annual_income', 'interaction_count', 'lead_score']

dv = DictVectorizer(sparse=False)

train_dict = df_train_copy[categorical_copy + numerical_copy].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dict = df_val_copy[categorical_copy + numerical_copy].to_dict(orient='records')
X_val = dv.transform(val_dict)

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
# solver='lbfgs' is the default solver in newer version of sklearn
# for older versions, you need to specify it explicitly
model.fit(X_train, y_train)

y_pred = model.predict_proba(X_val)[:, 1]
converted_decision = (y_pred >= 0.5)
no_courses_model_pred = round((y_val == converted_decision).mean(), 3)

no_courses_result = full_model_pred - no_courses_model_pred

print(no_courses_result)
print()
print(no_lead_result)

0.0020000000000000018

0.0030000000000000027


In [146]:
# Question 5 with all of the printouts

df_train_copy = df_train.copy()
df_val_copy = df_val.copy()
df_test_copy = df_test.copy()

del df_train_copy['interaction_count']
del df_val_copy['interaction_count']
del df_test_copy['interaction_count']

categorical_copy = categorical
numerical_copy = ['annual_income', 'number_of_courses_viewed', 'lead_score']

dv = DictVectorizer(sparse=False)

train_dict = df_train_copy[categorical_copy + numerical_copy].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dict = df_val_copy[categorical_copy + numerical_copy].to_dict(orient='records')
X_val = dv.transform(val_dict)

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
# solver='lbfgs' is the default solver in newer version of sklearn
# for older versions, you need to specify it explicitly
model.fit(X_train, y_train)

y_pred = model.predict_proba(X_val)[:, 1]
converted_decision = (y_pred >= 0.5)
no_interaction_model_pred = round((y_val == converted_decision).mean(), 3)

no_interaction_result = full_model_pred - no_interaction_model_pred

print(no_courses_result)
print()
print(no_lead_result)
print()
print(no_interaction_result)

0.0020000000000000018

0.0030000000000000027

0.04400000000000004


In [148]:
# Question 6
# C = .000001 

dv = DictVectorizer(sparse=False)

train_dict = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dict = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dict)

model = LogisticRegression(solver='liblinear', C=0.000001, max_iter=1000, random_state=42)
# solver='lbfgs' is the default solver in newer version of sklearn
# for older versions, you need to specify it explicitly
model.fit(X_train, y_train)

y_pred = model.predict_proba(X_val)[:, 1]
converted_decision = (y_pred >= 0.5)
model_pred = (y_val == converted_decision).mean()


print(model_pred)


0.598


In [149]:
# C = .00001 
dv = DictVectorizer(sparse=False)

train_dict = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dict = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dict)

model = LogisticRegression(solver='liblinear', C=0.00001, max_iter=1000, random_state=42)
# solver='lbfgs' is the default solver in newer version of sklearn
# for older versions, you need to specify it explicitly
model.fit(X_train, y_train)

y_pred = model.predict_proba(X_val)[:, 1]
converted_decision = (y_pred >= 0.5)
model_pred = (y_val == converted_decision).mean()


print(model_pred)

0.598


In [150]:
# C = .0001 
dv = DictVectorizer(sparse=False)

train_dict = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dict = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dict)

model = LogisticRegression(solver='liblinear', C=0.0001, max_iter=1000, random_state=42)
# solver='lbfgs' is the default solver in newer version of sklearn
# for older versions, you need to specify it explicitly
model.fit(X_train, y_train)

y_pred = model.predict_proba(X_val)[:, 1]
converted_decision = (y_pred >= 0.5)
model_pred = (y_val == converted_decision).mean()


print(model_pred)

0.613


In [151]:
# C = .001 
dv = DictVectorizer(sparse=False)

train_dict = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dict = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dict)

model = LogisticRegression(solver='liblinear', C=0.001, max_iter=1000, random_state=42)
# solver='lbfgs' is the default solver in newer version of sklearn
# for older versions, you need to specify it explicitly
model.fit(X_train, y_train)

y_pred = model.predict_proba(X_val)[:, 1]
converted_decision = (y_pred >= 0.5)
model_pred = (y_val == converted_decision).mean()


print(model_pred)

0.645
